## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-25-09-54-39 +0000,bbc,Energy bills to rise by 26% from January after...,https://www.bbc.co.uk/news/articles/cm20qnq33e...
1,2026-11-25-09-54-32 +0000,bbc,What will the energy cap changes mean for my b...,https://www.bbc.co.uk/news/articles/cdd29v8mp9...
2,2026-11-25-09-50-45 +0000,nypost,Illegal migrant murder suspect was released by...,https://nypost.com/2026/11/25/us-news/nyc-chin...
3,2026-11-25-09-48-05 +0000,bbc,Ministers ask public for views on changing age...,https://www.bbc.co.uk/news/articles/cwgkl31kmn...
4,2026-11-25-09-45-49 +0000,nypost,Alice Huang-linked firms listed $270 million i...,https://nypost.com/2026/11/25/business/alice-h...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2432/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
410,nvidia,33
286,trump,32
31,huang,22
38,china,16
627,jensen,15


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
118,2026-11-25-00-18-00 +0000,wsj,Jensen Huang will face a bipartisan maelstrom ...,https://www.wsj.com/politics/policy/jensen-hua...,145
271,2026-11-24-18-28-12 +0000,nypost,Trump says cheating Nvidia CEO Jensen Huang an...,https://nypost.com/2026/11/24/us-news/trump-sa...,131
368,2026-11-24-12-00-00 +0000,wsj,Secretary of State Marco Rubio sounded alarms ...,https://www.wsj.com/politics/policy/rubio-soun...,123
101,2026-11-25-01-12-13 +0000,nypost,"Marco Rubio’s concern about Jensen Huang, Nvid...",https://nypost.com/2026/11/24/us-news/rubio-je...,118
316,2026-11-24-15-40-14 +0000,nyt,"With Jensen Huang’s Return as C.E.O., Trump Ef...",https://www.nytimes.com/2026/11/23/technology/...,116


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
118,145,2026-11-25-00-18-00 +0000,wsj,Jensen Huang will face a bipartisan maelstrom ...,https://www.wsj.com/politics/policy/jensen-hua...
324,65,2026-11-24-15-25-29 +0000,nypost,Trump honors ‘America’s Mayor’ Rudy Giuliani w...,https://nypost.com/2026/11/24/us-news/trump-to...
98,52,2026-11-25-01-27-01 +0000,nypost,Cruise passenger heard crying night before she...,https://nypost.com/2026/11/24/us-news/cruise-p...
125,52,2026-11-24-23-49-03 +0000,nypost,Hudson River Trading expanding at 3 World Tra...,https://nypost.com/2026/11/24/business/hudson-...
81,48,2026-11-25-02-00-00 +0000,wsj,What would happen to the U.S. economy if the A...,https://www.wsj.com/economy/worse-than-the-dot...
66,46,2026-11-25-03-00-00 +0000,wsj,A Democratic push to enact Medicare for All is...,https://www.wsj.com/politics/policy/democrats-...
59,43,2026-11-25-03-18-52 +0000,nypost,Elderly Alzheimer-stricken pianist crosses 2 s...,https://nypost.com/2026/11/24/us-news/elderly-...
377,42,2026-11-24-11-28-30 +0000,nypost,Two suspects arrested after break-in reported ...,https://nypost.com/2026/11/24/world-news/princ...
28,40,2026-11-25-07-29-44 +0000,nyt,North Korea Kicks Out Swiss Diplomats in Spat ...,https://www.nytimes.com/2026/11/25/world/asia/...
54,38,2026-11-25-03-56-28 +0000,nypost,"Hackers target California city, potentially ex...",https://nypost.com/2026/11/24/us-news/el-cajon...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
